In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/northwind_processed.csv')
df['OrderDate'] = pd.to_datetime(df['OrderDate'])


#1. Dashboard de ventas mensuales con tendencias

sales_monthly = df.groupby('YearMonth')['TotalSales'].sum().reset_index()

fig1 = px.line(
    sales_monthly, 
    x='YearMonth', 
    y='TotalSales',
    title='1. Evolución de Ventas Mensuales con Tendencia',
    labels={'YearMonth': 'Mes/Año', 'TotalSales': 'Ventas ($)'},
    markers=True
)
fig1.update_traces(line_color='#1f77b4', line_width=2.5)
fig1.show()

#2. Mapa de calor de ventas por categoría y región
heatmap_data = df.pivot_table(
    index='CategoryName', 
    columns='CustomerRegion', 
    values='TotalSales', 
    aggfunc='sum', 
    fill_value=0
)

fig2 = px.imshow(
    heatmap_data,
    labels=dict(x="Región", y="Categoría", color="Ventas ($)"),
    title="2. Mapa de Calor: Ventas por Categoría y Región",
    text_auto='.2s',
    color_continuous_scale='Viridis'
)
fig2.show()

#3. Gráfico de barras: Top 10 clientes por volumen de compras

top_customers = df.groupby('CustomerName')['TotalSales'].sum().reset_index()
top_customers = top_customers.sort_values(by='TotalSales', ascending=False).head(10)

fig3 = px.bar(
    top_customers,
    x='TotalSales',
    y='CustomerName',
    orientation='h',
    title='3. Top 10 Clientes por Volumen de Compras',
    labels={'TotalSales': 'Total Comprado ($)', 'CustomerName': 'Cliente'},
    color='TotalSales',
    color_continuous_scale='Blues'
)
fig3.update_layout(yaxis={'categoryorder': 'total ascending'})
fig3.show()

#4. Línea de tiempo: Evolución de ventas trimestrales
df['YearQuarter'] = df['OrderYear'].astype(str) + "-Q" + df['OrderQuarter'].astype(str)
quarterly_sales = df.groupby('YearQuarter')['TotalSales'].sum().reset_index()

fig4 = px.line(
    quarterly_sales,
    x='YearQuarter',
    y='TotalSales',
    title='4. Evolución de Ventas Trimestrales',
    labels={'YearQuarter': 'Trimestre', 'TotalSales': 'Ventas ($)'},
    markers=True
)
fig4.update_traces(line_color='#2ca02c', line_dash='dash')
fig4.show()

#5. Treemap: Distribución de productos por categoría
prod_cat = df.groupby(['CategoryName', 'ProductName'])['TotalSales'].sum().reset_index()

fig5 = px.treemap(
    prod_cat,
    path=['CategoryName', 'ProductName'],
    values='TotalSales',
    title='5. Distribución de Productos y Categorías por Ventas',
    color='TotalSales',
    color_continuous_scale='RdBu'
)
fig5.show()

#6. Scatter plot: Relación entre descuentos y cantidades vendidas

fig6 = px.scatter(
    df,
    x='Discount',
    y='Quantity',
    size='TotalSales',
    color='CategoryName',
    hover_data=['ProductName'],
    title='6. Relación entre Porcentaje de Descuento y Cantidades Vendidas',
    labels={'Discount': 'Descuento (%)', 'Quantity': 'Unidades Vendidas'}
)
fig6.show()

#7. Gráfico circular: Distribución de países de clientes

country_dist = df.groupby('CustomerCountry')['CustomerID'].nunique().reset_index()
country_dist.columns = ['Country', 'CustomerCount']

fig7 = px.pie(
    country_dist,
    names='Country',
    values='CustomerCount',
    title='7. Distribución Geográfica de Clientes (Por País)',
    hole=0.4
)
fig7.show()

#8. Box plot: Análisis de precios por proveedor

top_suppliers = df['SupplierName'].value_counts().head(8).index
df_filtered_suppliers = df[df['SupplierName'].isin(top_suppliers)]

fig8 = px.box(
    df_filtered_suppliers,
    x='SupplierName',
    y='UnitPrice',
    color='SupplierName',
    title='8. Distribución y Dispersión de Precios Unitarios por Proveedor',
    labels={'SupplierName': 'Proveedor', 'UnitPrice': 'Precio Unitario ($)'}
)
fig8.update_layout(showlegend=False)
fig8.show()

#9. Gráfico de áreas: Acumulado de ventas por empleado

emp_sales = df.groupby(['YearMonth', 'EmployeeName'])['TotalSales'].sum().reset_index()

fig9 = px.area(
    emp_sales,
    x='YearMonth',
    y='TotalSales',
    color='EmployeeName',
    title='9. Venta Acumulada Mensual por Empleado',
    labels={'YearMonth': 'Fecha', 'TotalSales': 'Ventas ($)', 'EmployeeName': 'Empleado'}
)
fig9.show()

#10. Heatmap: Patrones temporales de pedidos

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['DayOfWeek'] = pd.Categorical(df['DayOfWeek'], categories=day_order, ordered=True)

temporal_pattern = df.pivot_table(
    index='DayOfWeek',
    columns='OrderMonth',
    values='OrderID',
    aggfunc='nunique',
    fill_value=0
)

fig10 = px.imshow(
    temporal_pattern,
    labels=dict(x="Mes del Año", y="Día de la Semana", color="Cantidad de Pedidos"),
    title="10. Patrón Temporal de Frecuencia de Pedidos",
    color_continuous_scale='YlGnBu'
)
fig10.show()

